# Prep: fastText embeddings + lexicon features with CatBoost

Lemmatised tokens (nlp-id), InSet positive/negative word counts, mean fastText `cc.id.300` vectors, and a GPU CatBoost classifier.

In [ ]:
DATA_DIR = "../data"
FASTTEXT_MODEL_PATH = "../models/cc.id.300.bin"

In [ ]:
import pandas as pd
import numpy as np
import nltk, string
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from stanza.pipeline.core import Pipeline
from nlp_id.stopword import StopWord
from nlp_id.lemmatizer import Lemmatizer
from nlp_id.tokenizer import Tokenizer
from nlp_id.postag import PosTag
from gensim.models.fasttext import load_facebook_model
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
import torch
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification, TFBertModel, TrainingArguments, Trainer, \
    AutoModelForSequenceClassification, EarlyStoppingCallback
# import evaluate

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score

In [ ]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
df = pd.read_csv(f"{DATA_DIR}/gemastik_22.csv")
df.head()

,id,sumber,kodekat,kodesubkat,kategori,subkategori,katakunci,tanggal,judul_berita,konten_berita,nama_tokoh,jabatan,organisasi,lokasi,alias,kutipan,sentimen_kutipan,sentimen_berita
0,568576e9e95,Detik,Q,Q,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jumlah pasien,2021-06-17,"BOR Sudah 89 Persen, RSUD Soreang Bandung Siap...",Bandung -\n\nSatuan Tugas Penanganan COVID-19 ...,[],['Bupati Bandung'],"['Bed Occupancy Ratio (BOR)', 'Pemda', 'BOR']","['Bandung', 'Kabupaten Bandung', 'Kota Bandung']",[],"['mengatakan, telah mengadakan rapat dan memut...",Netral,Netral
1,80f931fe95e,Detik,D,D2,Pengadaan Listrik dan Gas,Pengadaan Gas dan Produksi Es,Jaringan Gas,2021-03-18,"Honda CB650R Indonesia Dapat Penyegaran, Harga...",Jakarta -\n\nAstra Honda Motor (AHM) resmi men...,"['Biasa Mejeng', 'Resmi Dijual']",[],"['Big', 'LED']","['Jakarta', 'Indonesia']",[],"['""Rasa bangga itu menyempurnakan gaya hidup m...",Positif,Positif
2,1bd2af6ef23,Antara,C,C8,Industri Pengolahan,"Industri Kimia, Farmasi dan Obat Tradisional",farmasi,2021-02-23,Vaksin COVID-19 Pfizer disetujui satu dari tig...,Seoul (ANTARA) - Panel ahli pertama dari tiga ...,['Mulyo Sunyoto'],[],"['Universitas Oxford', 'BioNTech']","['Korea Selatan', 'Amerika Serikat', 'Korsel',...",['¬©'],[],Netral,Positif
3,ac2b55af24c,Detik,A,A2,"Pertanian, Kehutanan, dan Perikanan",Kehutanan dan Penebangan Kayu,bambu,2021-01-26,"TPU Srengseng Sawah Penuh, DKI Rujuk Jenazah C...","Jakarta - TPU Srengseng Sawah, Jakarta Selatan...",['Ivan Nurcahyo'],[],"['Pemprov DKI', 'Dinas Pertamanan']","['Jakarta', 'Jakarta Selatan', 'DKI Jakarta', ...",[],"['""Kita sebutnya TPU Bambu Wulung, sebenarnya ...",Positif,Positif
4,3f50747a1d4,Antara,Q,Q,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jasa Kesehatan dan Kegiatan Sosial Swasta,wabah penyakit,2021-05-04,Media Korea Utara sebut vaksin COVID-19 bukan ...,Seoul (ANTARA) - Media pemerintah Korea Utara ...,['Edwin Salvador'],[],['Partai Buruh'],"['Korea Utara', 'Korea Selatan', 'China', 'Kor...",['¬©'],['mengatakan wabah di sana tidak dapat dikesam...,Netral,Netral


In [ ]:
df['judul_konten'] = df['judul_berita'] + ' ' + df['konten_berita']
df['judul_konten'].head()

0    BOR Sudah 89 Persen, RSUD Soreang Bandung Siap...
1    Honda CB650R Indonesia Dapat Penyegaran, Harga...
2    Vaksin COVID-19 Pfizer disetujui satu dari tig...
3    TPU Srengseng Sawah Penuh, DKI Rujuk Jenazah C...
4    Media Korea Utara sebut vaksin COVID-19 bukan ...
Name: judul_konten, dtype: object

In [ ]:
nlp_stanza = Pipeline('id',
                      processors='tokenize,mwt,lemma',
                      tokenize_pretokenized=True,
                      )
stopwords = StopWord().get_stopword()
lemmatizer = Lemmatizer()
tokenizer = Tokenizer()
postagger = PosTag()

2023-09-11 02:09:43 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2023-09-11 02:09:43 INFO: Loading these models for language: id (Indonesian):
| Processor | Package |
-----------------------
| tokenize  | gsd     |
| mwt       | gsd     |
| lemma     | gsd     |

2023-09-11 02:09:43 INFO: Using device: cpu
2023-09-11 02:09:43 INFO: Loading: tokenize
2023-09-11 02:09:43 INFO: Loading: mwt
2023-09-11 02:09:43 INFO: Loading: lemma
2023-09-11 02:09:43 INFO: Done loading processors!
d:\miniconda3\envs\tf\lib\site-packages\sklearn\base.py:318: UserWarning: Trying to unpickle estimator DictVectorizer from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\miniconda3\envs\tf\lib\site-packages\sklearn\base.py:318: UserWarning: Trying to unpickle estimator DecisionTreeClassifier from version 0.22 when using version 1.2.2. This might lead to breaking code or

In [ ]:
def tokenize_lemmatize(text):
    lemmatized_doc = lemmatizer.lemmatize(text)
    return [word for word in tokenizer.tokenize(lemmatized_doc) if word not in stopwords]

In [ ]:
df['judul_konten_tokenized'] = df['judul_konten'].apply(tokenize_lemmatize)
df['judul_konten_tokenized'].head()

0    [bor, persen, rsud, soreang, bandung, kasur, b...
1    [honda, cbr, indonesia, segar, harga, rp, juta...
2    [vaksin, covid, pfizer, tuju, panel, ahli, kor...
3    [tpu, srengseng, sawah, penuh, dki, rujuk, jen...
4    [media, korea, utara, vaksin, covid, obat, muj...
Name: judul_konten_tokenized, dtype: object

In [ ]:
# kita muat kata-kata bersentiment tersebut ke memori
# dalam bentuk python's dictionary
# key = word
# value = sentiment score

def load_sentword(filename):
    file = open(filename, "r")
    file.readline()  # skip header
    sentword_score = {}
    for line in file:
        word, score = line.strip().split("\t")
        score = int(score)
        sentword_score[word] = score
    file.close()
    return sentword_score


sentword_score_neg = load_sentword(f"{DATA_DIR}/negative.tsv")
sentword_score_pos = load_sentword(f"{DATA_DIR}/positive.tsv")


# num positives or negatives
def num(text, sentword_score):
    count = 0
    for word in text:
        if word in sentword_score:
            count += 1
    return count


# num JJ positives or negatives
def num_jj(text, sentword_score):
    count = 0
    for word, pos in postagger.get_pos_tag(" ".join(text)):
        if pos == "JJ":
            if word in sentword_score:
                count += 1
    return count


# print(sentword_score)
print(sentword_score_neg['mencederai'])
print(sentword_score_pos['bagus'])

-4
2


In [ ]:
df["num_pos_judul_konten"] = df.judul_konten_tokenized. \
    map(lambda x: num(x, sentword_score_pos))

df["num_neg_judul_konten"] = df.judul_konten_tokenized. \
    map(lambda x: num(x, sentword_score_neg))

df["num_jj_pos_judul_konten"] = df.judul_konten_tokenized. \
    map(lambda x: num_jj(x, sentword_score_pos))

df["num_jj_neg_judul_konten"] = df.judul_konten_tokenized. \
    map(lambda x: num_jj(x, sentword_score_neg))

In [ ]:
ft_model = load_facebook_model(FASTTEXT_MODEL_PATH)

In [ ]:
ft_model.wv.most_similar('media', topn=10)

[('media-media', 0.7714970707893372),
 ('medianya', 0.757255494594574),
 ('mass-media', 0.7254214286804199),
 ('kemedia', 0.7241712212562561),
 ('socialmedia', 0.7184514403343201),
 ('sosialmedia', 0.7138491868972778),
 ('bermedia', 0.7018288969993591),
 ('Media', 0.7015385031700134),
 ('e-media', 0.7004505395889282),
 ('massmedia', 0.6973640322685242)]

In [ ]:
embedded_words = [[ft_model.wv[word] for word in token] for token in tqdm(df['judul_konten_tokenized'].tolist())]
embedded_words_encoding = [np.mean(embedded_word, axis=0) for embedded_word in tqdm(embedded_words)]
df_embedding = pd.DataFrame(embedded_words_encoding)
df_embedding.head()

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,0.000201,0.009318,0.014299,0.049614,-0.017443,-0.048011,-0.000149,0.020397,-0.016104,-0.079830,...,-0.004920,0.006627,-0.004757,-0.002436,0.004600,0.010463,0.022697,-0.023356,-0.008663,0.031868
1,0.021814,-0.012430,0.024316,0.021405,0.010416,-0.018267,-0.009480,-0.003020,-0.000080,-0.077585,...,0.002693,0.002037,-0.056673,-0.015800,-0.000507,0.027746,-0.007154,-0.017128,0.024032,0.047783
2,0.012353,0.011245,-0.001952,0.056266,0.004503,-0.048313,0.011655,0.010052,-0.009946,-0.064765,...,-0.019186,-0.015217,-0.011730,-0.016612,-0.004318,0.009394,-0.010254,-0.030489,0.011488,0.054460
3,0.002559,-0.016667,0.017217,0.053351,-0.019702,-0.071516,0.008987,-0.010791,-0.018234,-0.052556,...,-0.010164,-0.009626,0.007900,0.007081,0.024685,-0.004738,0.011917,-0.016638,-0.017436,0.039633
4,0.009578,0.003662,-0.000113,0.073117,0.008807,-0.066902,0.014115,-0.001011,-0.007999,-0.071712,...,-0.019018,-0.004231,-0.018451,-0.001973,-0.005696,-0.000949,-0.005335,-0.042493,-0.001537,0.047080


In [ ]:
def tsne_plot(for_word, w2v_model):
    # trained fastText model dimention
    dim_size = w2v_model.wv.vectors.shape[1]

    arrays = np.empty((0, dim_size), dtype='f')
    word_labels = [for_word]
    color_list = ['red']

    # adds the vector of the query word
    arrays = np.append(arrays, w2v_model.wv.__getitem__([for_word]), axis=0)

    # gets list of most similar words
    sim_words = w2v_model.wv.most_similar(for_word, topn=10)

    # adds the vector for each of the closest words to the array
    for wrd_score in sim_words:
        wrd_vector = w2v_model.wv.__getitem__([wrd_score[0]])
        word_labels.append(wrd_score[0])
        color_list.append('green')
        arrays = np.append(arrays, wrd_vector, axis=0)

    #---------------------- Apply PCA and tsne to reduce dimention --------------

    # fit 2d PCA model to the similar word vectors
    model_pca = PCA(n_components=10).fit_transform(arrays)

    # Finds 2d coordinates t-SNE
    np.set_printoptions(suppress=True)
    Y = TSNE(n_components=2, random_state=0, perplexity=15).fit_transform(model_pca)

    # Sets everything up to plot
    df_plot = pd.DataFrame({'x': [x for x in Y[:, 0]],
                            'y': [y for y in Y[:, 1]],
                            'words_name': word_labels,
                            'words_color': color_list})

    #------------------------- tsne plot Python -----------------------------------

    # plot dots with color and position
    plot_dot = sns.regplot(data=df_plot,
                           x="x",
                           y="y",
                           fit_reg=False,
                           marker="o",
                           scatter_kws={'s': 40,
                                        'facecolors': df_plot['words_color']
                                        }
                           )

    # Adds annotations with color one by one with a loop
    for line in range(0, df_plot.shape[0]):
        plot_dot.text(df_plot["x"][line],
                      df_plot['y'][line],
                      '  ' + df_plot["words_name"][line].title(),
                      horizontalalignment='left',
                      verticalalignment='bottom', size='medium',
                      color=df_plot['words_color'][line],
                      weight='normal'
                      ).set_size(15)

    plt.xlim(Y[:, 0].min() - 50, Y[:, 0].max() + 50)
    plt.ylim(Y[:, 1].min() - 50, Y[:, 1].max() + 50)

    plt.title('t-SNE visualization for word "{}'.format(for_word.title()) + '"')


In [ ]:
tsne_plot(for_word='media', w2v_model=ft_model)

ValueError: perplexity must be less than n_samples

In [ ]:
df_used = df[['num_pos_judul_konten', 'num_neg_judul_konten', 'num_jj_pos_judul_konten', 'num_jj_neg_judul_konten']]
df_merge = pd.concat([df_used.reset_index(drop=True), df_embedding], axis=1)
df_merge.head()

,num_pos_judul_konten,num_neg_judul_konten,num_jj_pos_judul_konten,num_jj_neg_judul_konten,0,1,2,3,4,5,...,290,291,292,293,294,295,296,297,298,299
0,29,48,5,3,0.000201,0.009318,0.014299,0.049614,-0.017443,-0.048011,...,-0.004920,0.006627,-0.004757,-0.002436,0.004600,0.010463,0.022697,-0.023356,-0.008663,0.031868
1,106,77,31,21,0.021814,-0.012430,0.024316,0.021405,0.010416,-0.018267,...,0.002693,0.002037,-0.056673,-0.015800,-0.000507,0.027746,-0.007154,-0.017128,0.024032,0.047783
2,29,29,2,3,0.012353,0.011245,-0.001952,0.056266,0.004503,-0.048313,...,-0.019186,-0.015217,-0.011730,-0.016612,-0.004318,0.009394,-0.010254,-0.030489,0.011488,0.054460
3,18,36,6,3,0.002559,-0.016667,0.017217,0.053351,-0.019702,-0.071516,...,-0.010164,-0.009626,0.007900,0.007081,0.024685,-0.004738,0.011917,-0.016638,-0.017436,0.039633
4,50,55,8,9,0.009578,0.003662,-0.000113,0.073117,0.008807,-0.066902,...,-0.019018,-0.004231,-0.018451,-0.001973,-0.005696,-0.000949,-0.005335,-0.042493,-0.001537,0.047080


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(df_merge, df['sentimen_berita'], test_size=0.2, random_state=42)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((4231, 304), (1058, 304), (4231,), (1058,))

In [ ]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier(task_type="GPU", verbose=1, random_state=42)
clf.fit(X_train, y_train,
        verbose=1,
        eval_set=(X_val, y_val),
        )

Learning rate set to 0.121647
0:	learn: 1.0602910	test: 1.0696149	best: 1.0696149 (0)	total: 29ms	remaining: 29s
1:	learn: 1.0291875	test: 1.0434554	best: 1.0434554 (1)	total: 39.8ms	remaining: 19.8s
2:	learn: 1.0024394	test: 1.0234437	best: 1.0234437 (2)	total: 50.2ms	remaining: 16.7s
3:	learn: 0.9796624	test: 1.0067902	best: 1.0067902 (3)	total: 59.2ms	remaining: 14.7s
4:	learn: 0.9611400	test: 0.9926030	best: 0.9926030 (4)	total: 69ms	remaining: 13.7s
5:	learn: 0.9449699	test: 0.9827962	best: 0.9827962 (5)	total: 78.4ms	remaining: 13s
6:	learn: 0.9290399	test: 0.9724465	best: 0.9724465 (6)	total: 86.9ms	remaining: 12.3s
7:	learn: 0.9154966	test: 0.9647479	best: 0.9647479 (7)	total: 95.7ms	remaining: 11.9s
8:	learn: 0.9037237	test: 0.9556798	best: 0.9556798 (8)	total: 104ms	remaining: 11.5s
9:	learn: 0.8932219	test: 0.9508807	best: 0.9508807 (9)	total: 113ms	remaining: 11.2s
10:	learn: 0.8839334	test: 0.9460221	best: 0.9460221 (10)	total: 122ms	remaining: 11s
11:	learn: 0.8755632	tes

In [ ]:
y_val_pred = clf.predict(X_val)
print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

     Negatif       0.58      0.38      0.45       173
      Netral       0.48      0.39      0.43       347
     Positif       0.65      0.80      0.72       538

    accuracy                           0.59      1058
   macro avg       0.57      0.52      0.53      1058
weighted avg       0.58      0.59      0.58      1058
